# OSOD pada dataset broiler (RGB) — Colab

Eksperimen A: training dari ImageNet ResNet50 (torchvision).

Legenda di judul tiap sel:
- **[ATUR]** — ubah nilai dulu sebelum run.
- **[RUN]** — tinggal run, tidak perlu diubah.
- **[OPSIONAL]** — boleh dilewati.

Urutan jalan: 1 (ATUR) → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9. Bagian "Visualisasi GT" di section 5 opsional.

Prasyarat di Google Drive (upload manual):
- Repo git berisi perubahan yang sudah di-commit dan di-push (`GIT_REPO_URL`, `GIT_REF`).
- `MyDrive/ayam/datasets/x/cctv/{train,val,test}/{rgb,label}/` — data RGB (thermal tidak dipakai). Path bisa diubah di `DATA_SRC`.

Catatan: dataset saat ini hanya sample (1 gambar per split). Dengan `BATCH_SIZE` > jumlah gambar train, sampler menghasilkan 0 iterasi.

## 1. Path dan hyperparameter — [ATUR] wajib diisi: DRIVE_ROOT, RUN_NAME, GIT_REPO_URL, GIT_REF, BATCH_SIZE, EPOCHS

In [ ]:
# [ATUR] isi path, git, dan hyperparameter sebelum run
# ---- path ----
DRIVE_ROOT    = '/content/drive/MyDrive/OSOD'
DATA_SRC      = '/content/drive/MyDrive/ayam/datasets/x/cctv'   # train/ val/ test/ {rgb,label}
RUN_NAME      = 'exp_A_imagenet_res50'
OUT_DIR       = DRIVE_ROOT + '/runs/' + RUN_NAME    # ckpt, logs, train.log, test.log, vis
# ---- git ----
GIT_REPO_URL  = 'https://github.com/pratamabintang/OSOD.git'
GIT_REF       = 'origin/main'                         # branch: origin/<nama>; commit: <sha>
GIT_CLEAN     = False                                 # True: hapus file untracked (kecuali data & output)
REPO_DIR      = '/content/OSOD'                      # ROOT_DIR repo; ../data = /content/data
PRETRAIN_DIR  = '/content/data/pretrain_imagenet_resnet50'

# ---- hyperparameter ----
NET            = 'res50'
SEEN           = 3          # 1 train, 2 test, 3 keduanya (1 kelas: selalu kelas broiler)
BATCH_SIZE     = 1          # harus <= jumlah gambar train (dengan flip, roidb x2)
NUM_WORKERS    = 2
EPOCHS         = 10
LR             = 0.001
LR_DECAY_STEP  = 4          # epoch
LR_DECAY_GAMMA = 0.1
OPTIMIZER      = 'sgd'
DISP_INTERVAL  = 10         # iterasi
CHECKPOINT_EPOCH_TO_TEST = None   # None = checkpoint terakhir

# ---- resume (eksperimen B, belum dipakai) ----
RESUME       = False
CHECKSESSION = 1
CHECKEPOCH   = 1
CHECKPOINT   = 0

print('OUT_DIR:', OUT_DIR)

## 2. Mount Drive dan cek input — [RUN] (muncul popup izin Google Drive)

In [ ]:
# [RUN]
from google.colab import drive
drive.mount('/content/drive')
import os
for d in [OUT_DIR, OUT_DIR + '/ckpt', OUT_DIR + '/logs', OUT_DIR + '/vis']:
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(DATA_SRC), 'DATA_SRC tidak ditemukan: ' + DATA_SRC
print('input OK')

## 3. Clone / reset repo git dan build `lib/` — [RUN] (build ±beberapa menit; ulang hanya jika repo berubah)

In [ ]:
# [RUN] clone atau reset ke GIT_REF
import os, shutil, subprocess

def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('gagal: ' + ' '.join(cmd) + '\n' + r.stderr)
    print(r.stdout.strip())

assert '<user>' not in GIT_REPO_URL, 'isi GIT_REPO_URL dulu'
if os.path.isdir(REPO_DIR) and not os.path.isdir(REPO_DIR + '/.git'):
    shutil.rmtree(REPO_DIR)
if not os.path.isdir(REPO_DIR):
    run(['git', 'clone', GIT_REPO_URL, REPO_DIR])
run(['git', 'fetch', '--all', '--prune'], cwd=REPO_DIR)
run(['git', 'reset', '--hard', GIT_REF], cwd=REPO_DIR)
if GIT_CLEAN:
    run(['git', 'clean', '-fd', '-e', 'datasets', '-e', 'models', '-e', 'logs', '-e', 'output', '-e', 'test_img'], cwd=REPO_DIR)
run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)

In [ ]:
%%bash
# [RUN] install dependensi dan build lib/ (±beberapa menit)
pip install -q easydict pyyaml pillow opencv-python scipy pandas matplotlib tensorboardX "cython<3"
cd /content/OSOD/lib && python setup.py build develop > /tmp/build.log 2>&1
echo BUILD_RC=$?
tail -n 5 /tmp/build.log

## 4. Pretrain ImageNet ResNet50 (torchvision) — [RUN] (unduh sekali, lewati jika file sudah ada)

Bobot dari URL resmi torchvision, dibungkus ke format yang dibaca `resnet.py`: `{'state_dict': {'module.<key>': tensor}}`, path `../data/pretrain_imagenet_resnet50/model_best.pth.tar` relatif terhadap `/content/OSOD`.

In [ ]:
# [RUN] unduh sekali, lewati jika file sudah ada
import torch, torchvision, os
os.makedirs(PRETRAIN_DIR, exist_ok=True)
dst = os.path.join(PRETRAIN_DIR, 'model_best.pth.tar')
if not os.path.exists(dst):
    tv = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
    sd = {'module.' + k: v for k, v in tv.state_dict().items()}
    torch.save({'state_dict': sd}, dst)
print('pretrain siap:', dst, os.path.getsize(dst), 'bytes')

## 5. Siapkan data (RGB saja) — [RUN] (cek jumlah gambar; error jika BATCH_SIZE > jumlah train)

In [ ]:
# [RUN] salin rgb/label ke repo; error jika BATCH_SIZE > jumlah train
import shutil, os
DST = REPO_DIR + '/datasets/broiler'
for split in ['train', 'val', 'test']:
    for sub in ['rgb', 'label']:
        dst_dir = os.path.join(DST, split, sub)
        if os.path.exists(dst_dir):
            shutil.rmtree(dst_dir)
        shutil.copytree(os.path.join(DATA_SRC, split, sub), dst_dir)

counts = {s: len(os.listdir(os.path.join(DST, s, 'rgb'))) for s in ['train', 'val', 'test']}
print('jumlah gambar:', counts)
assert BATCH_SIZE <= counts['train'], 'BATCH_SIZE (%d) > jumlah gambar train (%d)' % (BATCH_SIZE, counts['train'])

### Visualisasi GT — [OPSIONAL] [RUN] cek box ground truth sebelum training

Cek box ground truth di atas gambar sebelum training (konversi YOLO → px).

In [ ]:
# [OPSIONAL] [RUN] cek box GT sebelum training
import cv2, glob, os
import matplotlib.pyplot as plt

gt_dir = OUT_DIR + '/vis'
os.makedirs(gt_dir, exist_ok=True)
for split in ['train', 'val', 'test']:
    for p in sorted(glob.glob(REPO_DIR + '/datasets/broiler/' + split + '/rgb/*.png')):
        name = os.path.splitext(os.path.basename(p))[0]
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        H, W = im.shape[:2]
        with open(REPO_DIR + '/datasets/broiler/' + split + '/label/' + name + '.txt') as f:
            for line in f:
                c, cx, cy, w, h = map(float, line.split())
                x1, y1 = int((cx - w / 2) * W), int((cy - h / 2) * H)
                x2, y2 = int((cx + w / 2) * W), int((cy + h / 2) * H)
                cv2.rectangle(im, (x1, y1), (x2, y2), (255, 80, 0), 4)
        cv2.imwrite(gt_dir + '/gt_' + split + '_' + name + '.png', cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
        plt.figure(figsize=(8, 5))
        plt.imshow(im)
        plt.title('GT ' + split + ' / ' + name)
        plt.axis('off')
        plt.show()

## 6. Training — [RUN] (ATUR di sel 1: EPOCHS, LR, BATCH_SIZE; RESUME hanya eksperimen B)

Checkpoint ditulis langsung ke Drive lewat symlink `models/res50/broiler` → `OUT_DIR/ckpt`. Log TensorBoard ke `OUT_DIR/logs`.

In [ ]:
# [RUN] training; ATUR EPOCHS/LR/BATCH_SIZE di section 1. RESUME hanya eksperimen B
import subprocess, os
os.chdir(REPO_DIR)

os.makedirs(REPO_DIR + '/models/' + NET, exist_ok=True)
link_ckpt = REPO_DIR + '/models/' + NET + '/broiler'
if not os.path.lexists(link_ckpt):
    os.symlink(OUT_DIR + '/ckpt', link_ckpt)
link_log = REPO_DIR + '/logs'
if not os.path.lexists(link_log):
    os.symlink(OUT_DIR + '/logs', link_log)

cmd = [
    'python', '-u', 'trainval_net.py',
    '--dataset', 'broiler', '--net', NET,
    '--bs', str(BATCH_SIZE), '--nw', str(NUM_WORKERS),
    '--lr', str(LR), '--lr_decay_step', str(LR_DECAY_STEP),
    '--lr_decay_gamma', str(LR_DECAY_GAMMA), '--o', OPTIMIZER,
    '--epochs', str(EPOCHS), '--disp_interval', str(DISP_INTERVAL),
    '--save_dir', 'models', '--cuda', '--seen', str(SEEN),
]
if RESUME:
    cmd += ['--r', '--checksession', str(CHECKSESSION), '--checkepoch', str(CHECKEPOCH), '--checkpoint', str(CHECKPOINT)]

with open(OUT_DIR + '/train.log', 'w') as f:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
        f.write(line)
p.wait()
print('TRAIN_RC =', p.returncode)

## 7. Test — [RUN] (ATUR opsional: CHECKPOINT_EPOCH_TO_TEST, kosong = checkpoint terakhir)

Memilih checkpoint, menjalankan `test_net.py` dengan `--vis`, dan menyimpan log ke `OUT_DIR/test.log`.

In [ ]:
# [RUN] test; ATUR CHECKPOINT_EPOCH_TO_TEST (opsional, kosong = terakhir)
import glob, os, re, subprocess
os.chdir(REPO_DIR)

ckpts = sorted(glob.glob(OUT_DIR + '/ckpt/faster_rcnn_*.pth'), key=os.path.getmtime)
assert ckpts, 'tidak ada checkpoint di ' + OUT_DIR + '/ckpt'
if CHECKPOINT_EPOCH_TO_TEST is None:
    ck = ckpts[-1]
else:
    ck = [c for c in ckpts if re.search(r'_%d_\d+\.pth$' % CHECKPOINT_EPOCH_TO_TEST, c)][-1]
m = re.search(r'faster_rcnn_(\d+)_(\d+)_(\d+)\.pth$', ck)
s_, e_, p_ = m.group(1), m.group(2), m.group(3)
print('test checkpoint:', ck)

os.makedirs(REPO_DIR + '/test_img', exist_ok=True)
cmd = ['python', '-u', 'test_net.py', '--dataset', 'broiler', '--net', NET,
       '--s', s_, '--checkepoch', e_, '--p', p_,
       '--load_dir', 'models', '--cuda', '--seen', str(SEEN), '--vis']
out = subprocess.run(cmd, capture_output=True, text=True)
with open(OUT_DIR + '/test.log', 'w') as f:
    f.write(out.stdout + out.stderr)
print(out.stdout[-3000:])
print(out.stderr[-2000:])
print('TEST_RC =', out.returncode)

## 8. Visualisasi — [RUN] (grafik loss, AP, gambar deteksi)

Loss training dari `train.log`, AP dari `test.log`, dan gambar deteksi dari `test_img/`.

In [ ]:
# [RUN] grafik loss, AP, gambar deteksi
import re, os, shutil
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

steps, losses = [], []
pat = re.compile(r'\[epoch\s+(\d+)\]\[iter\s+(\d+)/\s*(\d+)\] loss: ([\d.]+)')
with open(OUT_DIR + '/train.log') as f:
    for line in f:
        m = pat.search(line)
        if m:
            e, i, n, l = int(m.group(1)), int(m.group(2)), int(m.group(3)), float(m.group(4))
            steps.append((e - 1) * n + i)
            losses.append(l)

if steps:
    plt.figure(figsize=(7, 4))
    plt.plot(steps, losses, marker='o')
    plt.xlabel('iterasi'); plt.ylabel('loss (rata-rata window)')
    plt.title('Loss training — ' + RUN_NAME)
    plt.grid(True, alpha=0.3)
    plt.savefig(OUT_DIR + '/loss.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('belum ada baris loss di train.log')

if os.path.exists(OUT_DIR + '/test.log'):
    with open(OUT_DIR + '/test.log') as f:
        ap_lines = [l.strip() for l in f if 'AP for' in l or 'Mean AP' in l]
    print('\n'.join(ap_lines) if ap_lines else 'belum ada baris AP di test.log')

vis_dir = REPO_DIR + '/test_img'
imgs = sorted(f for f in os.listdir(vis_dir) if f.endswith('.png')) if os.path.exists(vis_dir) else []
os.makedirs(OUT_DIR + '/vis', exist_ok=True)
for f in imgs:
    shutil.copy(os.path.join(vis_dir, f), os.path.join(OUT_DIR, 'vis', 'det_' + f))
print('gambar deteksi:', len(imgs))
for f in imgs[:5]:
    display(IPImage(filename=os.path.join(vis_dir, f), width=640))

## 9. Simpan hasil ke Drive — [RUN] (salin .pkl dan .xlsx ke Drive)

Checkpoint sudah di Drive lewat symlink. Di sini disalin hasil test (`.pkl` deteksi) dan `similarity_*.xlsx` jika ada.

In [ ]:
# [RUN] salin hasil ke Drive
import glob, shutil, os
for f in glob.glob(REPO_DIR + '/output/**/*.pkl', recursive=True) + glob.glob(REPO_DIR + '/similarity_*.xlsx'):
    shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))
print('tersimpan di', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))